# Neurotransmitter Probability Characterisation across Drosophila Connectome Nodules

The datasets should be downloaded into the data directory following the instructions on GitHub. 

## Install packages and configure environment

In [1]:
# Import external libraries
import dask
import numpy as np
import pyvista as pv
from dask.distributed import Client

# Import python libraries
import os

# Import local scripts
import brainz
import pipeline
import preprocess
import util

# Configure environment - I am assuming 8 cores are available
client = Client(processes=False, threads_per_worker=8)
dask.config.set({"dataframe.shuffle.method": "tasks"})
print(f"Dask Dashboard at {client.dashboard_link}")
preprocess.run() # ~10 mins first run
pv.set_jupyter_backend("client") 

MINSIZE = 30 # Minimum number of nodes in a cluster/community

C:\Users\cielb\.pyenv\pyenv-win\versions\3.13.0\Lib\contextlib.py:148: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.92s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Dask Dashboard at http://10.14.0.2:8787/status
Notice: this may take about 10 minutes if this is the first time running preprocess.py. 

08:15:37 Preprocessing complete!


## Load and Preview Combined Full Datasets

First, load in the main connectome file (proofread_connections_783.parquet, ~972 MB). Then attach coordinates to edges using the coordinate data in the coordinate file (flywire_synapses_783.parquet, ~12.7 GB). Finally, normalise neurotransmitter probabilities by grouping 'other' neurotransmitter probabilities into an 'other' column and ensuring the probabilities for each neural connection/edge sum to 1 (they are off by a few decimal points), then preview the first few rows of the merged datasets.

In [2]:
connectome = pipeline.load_connectome("data/proofread_connections_783.parquet")
connectome = pipeline.attach_coords(connectome)
connectome = pipeline.normalise_nt_probs(connectome)
connectome.head(5)

08:15:56 Loading connectome ...
08:15:56 Connectome loaded
08:15:56 Loading coordinate file ...
08:15:57 Loaded coordinate file
08:15:57 Attaching coordinates ...
08:15:57 Coordinates attached


x  \
pre                post               syn_count neuropil gaba     ach      glut     oct      ser      da                    
720575940600782252 720575940640291125 1         ME_R     0.051772 0.009158 0.934698 0.003996 0.000180 0.000196  1120786.0   
720575940600934665 720575940611314602 2         LOP_R    0.009039 0.842430 0.026683 0.103866 0.000192 0.017789  1129999.0   
                   720575940620807919 1         LOP_R    0.090921 0.030018 0.850171 0.027256 0.000469 0.001164  1128834.0   
720575940601067436 720575940632789924 3         ME_R     0.008159 0.854459 0.030875 0.080715 0.001282 0.024510  1227736.0   
720575940601423625 720575940636570725 1         LO_R     0.002152 0.989977 0.000110 0.004439 0.000003 0.003321  1078427.0   

                                                                                                                       y  \
pre                post               syn_count neuropil gaba     ach      glut     oct      ser      da                   
720575940600782252 720575940640291125 1         ME_R     0.051772 0.009158 0.934698 0.003996 0.000180 0.000196  510896.0   
720575940600934665 720575940611314602 2         LOP_R    0.009039 0.842430 0.026683 0.103866 0.000192 0.017789  429994.0   
                   720575940620807919 1         LOP_R    0.090921 0.030018 0.850171 0.027256 0.000469 0.001164  430604.0   
720575940601067436 720575940632789924 3         ME_R     0.008159 0.854459 0.030875 0.080715 0.001282 0.024510  532230.0   
720575940601423625 720575940636570725 1         LO_R     0.002152 0.989977 0.000110 0.004439 0.000003 0.003321  474431.0   

                                                                                                                            z  
pre                post               syn_count neuropil gaba     ach      glut     oct      ser      da                       
720575940600782252 720575940640291125 1         ME_R     0.051772 0.009158 0.934698 0.003996 0.000180 0.000196  278480.000000  
720575940600934665 720575940611314602 2         LOP_R    0.009039 0.842430 0.026683 0.103866 0.000192 0.017789  321350.000000  
                   720575940620807919 1         LOP_R    0.090921 0.030018 0.850171 0.027256 0.000469 0.001164  320940.000000  
720575940601067436 720575940632789924 3         ME_R     0.008159 0.854459 0.030875 0.080715 0.001282 0.024510  270146.666667  
720575940601423625 720575940636570725 1         LO_R     0.002152 0.989977 0.000110 0.004439 0.000003 0.003321  237680.000000

## Isolate Nodule Neurons from Linker Neurons

The original premise of this project was to use a modified girvan-newman algorithm to remove neurons in 'bridge'/'linker' regions from the dataset by using the fact they would have relatively high edge betweenness. However, this is impractical (trust me, I tried to make it work for almost a month). Here I visualise the initial output generated from hdbscan clustering on xyz coordinates on a small sample of the drosophila connectome, then apply clustering on the full connectome dataset.

In [ ]:
# Visualise small connectome after clustering xyz coordinates
import util
small = pipeline.load_connectome("data/small.parquet")
small = pipeline.attach_coords(small)
small = util.do_hdbscan(small, MINSIZE)
plotter = brainz.get_plotter(small, "hdbscan_id")
brainz.save(plotter, "results/small")
plotter.show()

In [ ]:
# Cluster full connectome
